In [6]:
### Chunk 1: Imports & Data Loading / Scaling

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold

# --- Load & scale training data ---
train_df = pd.read_csv(
    'C:/Users/dash0006/OneDrive - University of Oklahoma/'
    'MVGPR_SS_Nano_p2/Python_code/Data/dat_bin_odd_t.csv'
)
X_train = train_df[['Mo', 'Nb', 'Ta', 'V', 'W']].values.astype(np.float32)
y_train = train_df[
    ['var_log_max_strain', 'var_bc_1', 'var_bc_2', 'var_bc_3', 'var_bc_4']
].values.astype(np.float32)

input_scaler = MinMaxScaler(feature_range=(0, 1))
output_scaler = MinMaxScaler(feature_range=(0, 1))
X_train_scaled = input_scaler.fit_transform(X_train)
y_train_scaled = output_scaler.fit_transform(y_train)

In [7]:
### Chunk 2 (revised): BranchNet definition

import torch.nn as nn
import torch

class BranchNet(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        total_layers: int,
        dropout_rate: float,
        output_size: int
    ):
        super().__init__()
        # allocate layers
        if total_layers <= 4:
            log_layers  = 2
            rest_layers = total_layers - log_layers
        elif total_layers == 5:
            log_layers  = 3
            rest_layers = 2
        else:  # total_layers == 6
            log_layers  = 3
            rest_layers = 3

        # build the "log_max_strain" branch (1 output)
        layers_log = []
        for i in range(log_layers):
            in_feats = input_size if i == 0 else hidden_size
            layers_log += [
                nn.Linear(in_feats, hidden_size),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ]
        layers_log.append(nn.Linear(hidden_size, 1))
        self.branch_log = nn.Sequential(*layers_log)

        # build the "rest" branch (output_size-1 outputs)
        layers_rest = []
        for i in range(rest_layers):
            in_feats = input_size if i == 0 else hidden_size
            layers_rest += [
                nn.Linear(in_feats, hidden_size),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ]
        layers_rest.append(nn.Linear(hidden_size, output_size - 1))
        self.branch_rest = nn.Sequential(*layers_rest)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out_log  = self.branch_log(x)    # [batch, 1]
        out_rest = self.branch_rest(x)   # [batch, output_size-1]
        return torch.cat([out_log, out_rest], dim=1)


In [8]:
### Chunk 3 (revised): 4-D CV grid search with BranchNet

from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
import numpy as np
import torch

def grid_search_full_branch(
    X, y,
    layer_list, neuron_list,
    lr_list, batch_size_list,
    num_epochs=20, n_splits=5
):
    # stratify on the primary target (m_log_max_strain)
    bins = pd.qcut(y[:, 0], q=n_splits, labels=False, duplicates='drop')
    skf  = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    best = {'val_loss': float('inf')}
    results = []

    for total_layers in layer_list:
        for nh in neuron_list:
            for lr in lr_list:
                for bs in batch_size_list:
                    fold_losses = []
                    print(f"\nTesting → layers={total_layers}, neurons={nh}, lr={lr}, batch_size={bs}")
                    for tr_idx, val_idx in skf.split(X, bins):
                        X_tr, y_tr = X[tr_idx], y[tr_idx]
                        X_val, y_val = X[val_idx], y[val_idx]

                        tr_ds     = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
                        tr_loader = DataLoader(tr_ds, batch_size=bs, shuffle=True)

                        model     = BranchNet(
                            input_size = X.shape[1],
                            hidden_size = nh,
                            total_layers = total_layers,
                            dropout_rate = 0.2,
                            output_size = y.shape[1]
                        )
                        criterion = nn.MSELoss()
                        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

                        # train one fold
                        model.train()
                        for _ in range(num_epochs):
                            for xb, yb in tr_loader:
                                optimizer.zero_grad()
                                loss = criterion(model(xb), yb)
                                loss.backward()
                                optimizer.step()

                        # val
                        model.eval()
                        with torch.no_grad():
                            val_preds = model(torch.tensor(X_val))
                            loss_val  = criterion(val_preds, torch.tensor(y_val)).item()
                        fold_losses.append(loss_val)

                    avg_loss = np.mean(fold_losses)
                    print(f"  → Avg val loss: {avg_loss:.6f}")

                    results.append({
                        'layers': total_layers,
                        'neurons': nh,
                        'lr': lr,
                        'batch_size': bs,
                        'val_loss': avg_loss
                    })
                    if avg_loss < best['val_loss']:
                        best = {
                            'layers': total_layers,
                            'neurons': nh,
                            'lr': lr,
                            'batch_size': bs,
                            'val_loss': avg_loss
                        }

    print(
        f"\nBest config → layers={best['layers']}, neurons={best['neurons']}, "
        f"lr={best['lr']}, batch_size={best['batch_size']} "
        f"(val_loss={best['val_loss']:.6f})"
    )
    return best, results

# then invoke with
layer_list      = [3, 4, 5, 6]
neuron_list     = [8, 16, 32, 64]
lr_list         = [0.001, 0.005, 0.01, 0.05, 0.1]
batch_size_list = [4, 8, 16, 32, 50]

best_cfg, all_results = grid_search_full_branch(
    X_train_scaled, y_train_scaled,
    layer_list, neuron_list,
    lr_list, batch_size_list,
    num_epochs=20,
    n_splits=5
)


Testing → layers=3, neurons=8, lr=0.001, batch_size=4
  → Avg val loss: 0.038462

Testing → layers=3, neurons=8, lr=0.001, batch_size=8
  → Avg val loss: 0.041017

Testing → layers=3, neurons=8, lr=0.001, batch_size=16
  → Avg val loss: 0.071590

Testing → layers=3, neurons=8, lr=0.001, batch_size=32
  → Avg val loss: 0.066966

Testing → layers=3, neurons=8, lr=0.001, batch_size=50
  → Avg val loss: 0.095898

Testing → layers=3, neurons=8, lr=0.005, batch_size=4
  → Avg val loss: 0.034494

Testing → layers=3, neurons=8, lr=0.005, batch_size=8
  → Avg val loss: 0.034535

Testing → layers=3, neurons=8, lr=0.005, batch_size=16
  → Avg val loss: 0.034322

Testing → layers=3, neurons=8, lr=0.005, batch_size=32
  → Avg val loss: 0.037260

Testing → layers=3, neurons=8, lr=0.005, batch_size=50
  → Avg val loss: 0.044770

Testing → layers=3, neurons=8, lr=0.01, batch_size=4
  → Avg val loss: 0.035405

Testing → layers=3, neurons=8, lr=0.01, batch_size=8
  → Avg val loss: 0.033167

Testing → l

In [9]:
def train_model_adam(model, optimizer, criterion, dataloader, num_epochs):
    model.train()
    for epoch in range(num_epochs):
        epoch_losses = []
        for X_batch, y_batch in dataloader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
        avg = np.mean(epoch_losses)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg:.6f}")
    return avg



### Chunk 4: Train final BranchNet on full data

# unpack best config
nl = best_cfg['layers']
nh = best_cfg['neurons']
lr = best_cfg['lr']
bs = best_cfg['batch_size']

# build final model
final_model = BranchNet(
    input_size   = X_train_scaled.shape[1],
    hidden_size  = nh,
    total_layers = nl,
    dropout_rate = 0.2,
    output_size  = y_train_scaled.shape[1]
)

optimizer = torch.optim.Adam(final_model.parameters(), lr=lr)
criterion = nn.MSELoss()

# full-data loader
full_ds     = TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(y_train_scaled, dtype=torch.float32)
)
full_loader = DataLoader(full_ds, batch_size=bs, shuffle=True)

print("\nTraining final model on full training set...")
final_loss = train_model_adam(final_model, optimizer, criterion, full_loader, num_epochs=20)
print(f"Final training loss: {final_loss:.6f}")



Training final model on full training set...
Epoch 1/20, Loss: 0.065109
Epoch 2/20, Loss: 0.051097
Epoch 3/20, Loss: 0.044222
Epoch 4/20, Loss: 0.039347
Epoch 5/20, Loss: 0.036775
Epoch 6/20, Loss: 0.037191
Epoch 7/20, Loss: 0.036407
Epoch 8/20, Loss: 0.037279
Epoch 9/20, Loss: 0.038860
Epoch 10/20, Loss: 0.034438
Epoch 11/20, Loss: 0.033651
Epoch 12/20, Loss: 0.033453
Epoch 13/20, Loss: 0.032952
Epoch 14/20, Loss: 0.034298
Epoch 15/20, Loss: 0.033887
Epoch 16/20, Loss: 0.033513
Epoch 17/20, Loss: 0.033260
Epoch 18/20, Loss: 0.032582
Epoch 19/20, Loss: 0.032045
Epoch 20/20, Loss: 0.034269
Final training loss: 0.034269


In [10]:
import os
from sklearn.metrics import r2_score, mean_squared_error

# Directory and test files
data_dir = 'C:/Users/dash0006/OneDrive - University of Oklahoma/MVGPR_SS_Nano_p2/Python_code/Data/May-Results/branching'
test_files = [
    'dat_test_bin_even_t.csv',
    'dat_test_qq_t.csv'
]

input_cols = ['Mo', 'Nb', 'Ta', 'V', 'W']
output_names = ['var_log_max_strain', 'var_bc_1', 'var_bc_2', 'var_bc_3', 'var_bc_4']

for fname in test_files:
    path = os.path.join(data_dir, fname)
    df = pd.read_csv(path)
    
    # Extract and scale inputs
    X_test = df[input_cols].values.astype(np.float32)
    X_test_scaled = input_scaler.transform(X_test)
    X_test_tensor = torch.tensor(X_test_scaled)
    
    # Predict (scaled) and invert transform
    final_model.eval()
    with torch.no_grad():
        preds_scaled = final_model(X_test_tensor).numpy()
    preds = output_scaler.inverse_transform(preds_scaled)
    
    # Save inputs + predictions
    pred_df = df[input_cols].copy()
    for i, name in enumerate(output_names):
        pred_df[name] = preds[:, i]
    out_fname = f'Vpredictions_{os.path.splitext(fname)[0]}.csv'
    out_path = os.path.join(data_dir, out_fname)
    pred_df.to_csv(out_path, index=False)
    print(f"Saved predictions to: {out_path}")
    
    # Compute metrics against the true outputs in df
    actual = df[output_names].values.astype(np.float32)
    mask = ~np.isnan(actual).any(axis=1)
    actual_clean, preds_clean = actual[mask], preds[mask]
    
    print(f"\nMetrics for {fname}:")
    for i, name in enumerate(output_names):
        r2 = r2_score(actual_clean[:, i], preds_clean[:, i])
        rmse = mean_squared_error(actual_clean[:, i], preds_clean[:, i], squared=False)
        print(f"  {name}: R² = {r2:.4f}, RMSE = {rmse:.4f}")
    print("-" * 40)

Saved predictions to: C:/Users/dash0006/OneDrive - University of Oklahoma/MVGPR_SS_Nano_p2/Python_code/Data/May-Results/branching\Vpredictions_dat_test_bin_even_t.csv

Metrics for dat_test_bin_even_t.csv:
  var_log_max_strain: R² = 0.0416, RMSE = 0.0057
  var_bc_1: R² = -0.1679, RMSE = 0.0037
  var_bc_2: R² = -0.1050, RMSE = 0.0014
  var_bc_3: R² = 0.0162, RMSE = 0.0648
  var_bc_4: R² = -0.3975, RMSE = 0.0221
----------------------------------------
Saved predictions to: C:/Users/dash0006/OneDrive - University of Oklahoma/MVGPR_SS_Nano_p2/Python_code/Data/May-Results/branching\Vpredictions_dat_test_qq_t.csv

Metrics for dat_test_qq_t.csv:
  var_log_max_strain: R² = -0.1311, RMSE = 0.0018
  var_bc_1: R² = -0.4294, RMSE = 0.0033
  var_bc_2: R² = -0.1835, RMSE = 0.0016
  var_bc_3: R² = -0.0286, RMSE = 0.0231
  var_bc_4: R² = -0.0851, RMSE = 0.0999
----------------------------------------


C:\Users\dash0006\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\dash0006\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\dash0006\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\dash0006\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will b